# 06 - SQL Analytics

## Objective

Perform SQL-based HR analytics on employee attrition risk data.

## Business Goal

Answer key business questions:

- Which departments have highest attrition risk?
- Which job roles have highest attrition risk?
- What is average risk score by department?
- Which employees require immediate retention action?

## 1. Load Risk Scoring Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned/employee_attrition_risk_scores.csv")

df.head()

,EmployeeNumber,Department,JobRole,Age,MonthlyIncome,YearsAtCompany,Actual_Attrition,Attrition_Probability,Risk_Level
0,1495,Sales,Sales Representative,24,2033,1,0,0.292936,Low
1,1246,Research & Development,Research Scientist,44,2011,10,0,0.020328,Low
2,613,Sales,Manager,31,11557,5,0,0.309931,Medium
3,1288,Research & Development,Manager,44,19190,25,0,0.048767,Low
4,90,Research & Development,Research Scientist,36,3388,1,1,0.651570,High


## 2. Create SQLite Database Connection

In [2]:
import sqlite3

conn = sqlite3.connect(":memory:")

## 3. Load Dataset into SQL Table

In [3]:
df.to_sql(
    "employee_risk",
    conn,
    index=False,
    if_exists="replace"
)

print("Table created successfully")

Table created successfully


## 4. View Sample Records

In [4]:
query = """
SELECT *
FROM employee_risk
LIMIT 5
"""

pd.read_sql(query, conn)

,EmployeeNumber,Department,JobRole,Age,MonthlyIncome,YearsAtCompany,Actual_Attrition,Attrition_Probability,Risk_Level
0,1495,Sales,Sales Representative,24,2033,1,0,0.292936,Low
1,1246,Research & Development,Research Scientist,44,2011,10,0,0.020328,Low
2,613,Sales,Manager,31,11557,5,0,0.309931,Medium
3,1288,Research & Development,Manager,44,19190,25,0,0.048767,Low
4,90,Research & Development,Research Scientist,36,3388,1,1,0.651570,High


## Business Question 1

Which departments have the highest average attrition risk?

In [5]:
query = """
SELECT
    Department,
    ROUND(AVG(Attrition_Probability), 3) AS Avg_Risk
FROM employee_risk
GROUP BY Department
ORDER BY Avg_Risk DESC
"""

pd.read_sql(query, conn)

,Department,Avg_Risk
0,Sales,0.442
1,Human Resources,0.341
2,Research & Development,0.297


## Business Question 2

Which job roles have the highest average attrition risk?

In [6]:
query = """
SELECT
    JobRole,
    ROUND(AVG(Attrition_Probability), 3) AS Avg_Risk
FROM employee_risk
GROUP BY JobRole
ORDER BY Avg_Risk DESC
"""

pd.read_sql(query, conn)

,JobRole,Avg_Risk
0,Sales Representative,0.690
1,Laboratory Technician,0.428
2,Human Resources,0.391
3,Sales Executive,0.370
4,Research Scientist,0.334
5,Manufacturing Director,0.272
6,Healthcare Representative,0.204
7,Manager,0.155
8,Research Director,0.049


## Business Question 3

How many employees belong to each risk category?

In [7]:
query = """
SELECT
    Risk_Level,
    COUNT(*) AS Employee_Count
FROM employee_risk
GROUP BY Risk_Level
ORDER BY Employee_Count DESC
"""

pd.read_sql(query, conn)

,Risk_Level,Employee_Count
0,Low,161
1,Medium,71
2,High,62


## Business Question 4

Identify High Risk Employees requiring retention action.

In [8]:
query = """
SELECT
    EmployeeNumber,
    Department,
    JobRole,
    Age,
    MonthlyIncome,
    Attrition_Probability
FROM employee_risk
WHERE Risk_Level = 'High'
ORDER BY Attrition_Probability DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,EmployeeNumber,Department,JobRole,Age,MonthlyIncome,Attrition_Probability
0,478,Sales,Sales Representative,21,2174,0.992223
1,1273,Sales,Sales Representative,25,1118,0.988337
2,959,Sales,Sales Representative,19,2121,0.965704
3,1318,Sales,Sales Executive,40,9094,0.965595
4,1439,Sales,Sales Representative,25,4400,0.965358
5,1053,Research & Development,Research Scientist,26,2042,0.953108
6,175,Sales,Sales Executive,31,4559,0.933650
7,702,Research & Development,Research Scientist,33,3348,0.932043
8,720,Sales,Sales Executive,24,4577,0.916614
9,424,Human Resources,Human Resources,31,6410,0.916083


## Business Question 5

Which departments contain the highest number of High Risk employees?

In [9]:
query = """
SELECT
    Department,
    COUNT(*) AS High_Risk_Count
FROM employee_risk
WHERE Risk_Level = 'High'
GROUP BY Department
ORDER BY High_Risk_Count DESC
"""

pd.read_sql(query, conn)

,Department,High_Risk_Count
0,Research & Development,33
1,Sales,26
2,Human Resources,3


## SQL Analytics Summary

### Key Findings

- Identify highest-risk departments.
- Identify highest-risk job roles.
- Quantify employee risk distribution.
- Prioritize employees for retention initiatives.

### Business Impact

Provides HR leaders with data-driven retention insights and workforce planning support.